# Day 3: Batting and Bowling Player-Match Dataset Construction

## Project
**IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning**

## Day 3 objective

Convert match-level source data into two analytical datasets:

1. **Batting player-match dataset:** one row per batter per match.
2. **Bowling player-match dataset:** one row per bowler per match.

Each row must describe a player's performance in one match, together with the match context needed for later EDA, SQL, feature engineering, ML, DL, Django APIs, and Power BI.

### Source-of-truth policy

- Cricsheet JSON is the primary historical source.
- The 2026 CSV scorecards are a comparison/reference source.
- Do not append the 2026 scorecards to the historical JSON outputs automatically. That could double-count matches already present in Cricsheet.
- Preserve original player and team names alongside normalized comparison fields.
- Do not invent player IDs, match dates, venues, or performance values.
- This notebook does not create next-match targets yet. Target construction belongs to Day 4.
- This notebook does not train ML/DL models.


## 1. Dataset design

### Batting dataset

One row represents one batter's observed batting performance in one match.

Expected fields include:

| Field | Meaning |
|---|---|
| `match_id` | Stable source match identifier |
| `date` | Match date |
| `season` | Source season |
| `venue`, `city` | Match location, where available |
| `team` | Batter's batting team |
| `opposition` | Other team in the match |
| `player_name` | Source player name |
| `player_id` | Cricsheet registry identifier, when available |
| `runs` | Runs credited to the batter |
| `balls_faced` | Balls faced under the documented delivery rule |
| `fours`, `sixes` | Boundaries hit by the batter |
| `dismissal_kind` | Dismissal type if recorded |
| `was_dismissed` | Whether a dismissal record exists for this batter |
| `batting_position` | Derived order of first appearance in the innings |

### Bowling dataset

One row represents one bowler's observed bowling performance in one match.

Expected fields include:

| Field | Meaning |
|---|---|
| `match_id`, `date`, `season` | Match identity and timing |
| `team`, `opposition` | Bowling team and opposing team |
| `player_name`, `player_id` | Bowler identity |
| `legal_balls` | Count of legal deliveries bowled |
| `overs` | Display overs derived from legal balls |
| `runs_conceded` | Batter runs plus wides and no-balls |
| `wickets` | Wickets credited to the bowler |
| `wides`, `noballs` | Bowling extras |
| `dots` | Legal deliveries with zero total runs off the delivery |

**Important:** Cricket overs are not decimal fractions. For example, 3 overs and 2 balls is displayed as `3.2`, not as `3.2` decimal overs. Store `legal_balls` as the reliable numeric measure and derive display overs from it.


## 2. Important cricket-data rules

The code uses explicit aggregation rules. These rules must be checked against the source and documented.

### Batting
- `runs` sums the `runs.batter` value for deliveries faced by the batter.
- `balls_faced` excludes wides and no-balls in this implementation.
- `fours` and `sixes` count batter-scored 4s and 6s where the run is not marked as a non-boundary run.
- `batting_position` is the order in which each batter first appears at the crease within the innings.
- A player can appear in more than one innings in unusual match formats, so the raw aggregation is first made at innings level and then rolled up to player-match level.

### Bowling
- `runs_conceded` = batter runs + wides + no-balls.
- Byes and leg-byes are not charged to the bowler.
- Bowler-credit wickets exclude `run out`, `retired hurt`, `obstructing the field`, and other non-bowler-credit dismissals according to the explicit list in the code.
- `legal_balls` excludes wides and no-balls.
- `dots` counts legal deliveries where the delivery total is zero.

These definitions can differ from some scorecard conventions. The notebook keeps the calculations explicit so they can be validated against the supplied scorecards.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CRICSHEET_DIR = RAW_DIR / "ipl_male_json"
DATA_2026_DIR = RAW_DIR / "2026"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

assert CRICSHEET_DIR.exists(), f"Missing extracted Cricsheet folder: {CRICSHEET_DIR}"
assert DATA_2026_DIR.exists(), f"Missing extracted 2026 folder: {DATA_2026_DIR}"

json_files = sorted(CRICSHEET_DIR.glob("*.json"))
assert json_files, f"No Cricsheet JSON files found in {CRICSHEET_DIR}"

print("Project root:", PROJECT_ROOT)
print("Cricsheet match files:", len(json_files))
print("2026 source folder:", DATA_2026_DIR)
print("Output folder:", PROCESSED_DIR)


Project root: d:\Git\Personal\ipl-player-performance
Cricsheet match files: 1243
2026 source folder: d:\Git\Personal\ipl-player-performance\data\raw\2026
Output folder: d:\Git\Personal\ipl-player-performance\data\processed


## 3. Helper functions

The helpers below standardize only fields used for comparison. Original source names remain in the datasets. Player IDs are taken from Cricsheet's registry mapping when the source provides one.


In [2]:
def normalize_label(value):
    """Comparison key only. Keep original source labels in separate columns."""
    if value is None or pd.isna(value):
        return None
    return re.sub(r"[^a-z0-9]", "", str(value).strip().lower())


def first_date(info):
    dates = info.get("dates", [])
    if not dates:
        return pd.NaT
    return pd.to_datetime(str(dates[0]), errors="coerce")


def match_context(match, path):
    info = match.get("info", {})
    teams = info.get("teams", [])
    registry_people = info.get("registry", {}).get("people", {})

    return {
        "match_id": path.stem,
        "source_file": path.name,
        "date": first_date(info),
        "season": str(info.get("season")) if info.get("season") is not None else None,
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team_1": teams[0] if len(teams) > 0 else None,
        "team_2": teams[1] if len(teams) > 1 else None,
        "match_type": info.get("type"),
        "registry_people": registry_people,
    }


def player_id_for(player_name, registry_people):
    """Return only the source registry ID; never fabricate an ID."""
    return registry_people.get(player_name)


def other_team(team, team_1, team_2):
    if team == team_1:
        return team_2
    if team == team_2:
        return team_1
    return None


def is_non_boundary(delivery):
    extras = delivery.get("extras", {}) or {}
    # Cricsheet can mark a batter-scored boundary as non-boundary in rare cases.
    return bool(extras.get("non_boundary", False))


## 4. Build batting and bowling player-match records from Cricsheet

The source is processed one match at a time to avoid loading the entire raw archive into memory. The intermediate aggregation key is match + innings + team + player. A second aggregation combines innings into one player-match row.


In [3]:
batting_innings_rows = []
bowling_innings_rows = []
match_rows = []
player_registry_rows = []

# Dismissal kinds that are generally not credited as bowler wickets.
NON_BOWLER_WICKET_KINDS = {
    "run out",
    "retired hurt",
    "retired out",
    "obstructing the field",
}

for file_number, path in enumerate(json_files, start=1):
    with path.open("r", encoding="utf-8") as f:
        match = json.load(f)

    context = match_context(match, path)
    match_rows.append({
        key: value for key, value in context.items()
        if key != "registry_people"
    })

    registry_people = context["registry_people"]

    for team_name, players in match.get("info", {}).get("players", {}).items():
        for player_name in players:
            player_registry_rows.append({
                "player_name": player_name,
                "player_name_norm": normalize_label(player_name),
                "team_name_from_roster": team_name,
                "player_id": player_id_for(player_name, registry_people),
                "source_match_id": context["match_id"],
            })

    for innings_number, innings in enumerate(match.get("innings", []), start=1):
        batting_team = innings.get("team")
        opposition = other_team(
            batting_team,
            context["team_1"],
            context["team_2"]
        )

        batting_stats = {}
        bowling_stats = {}
        first_seen_batter = {}
        wicket_details = {}

        legal_ball_sequence = 0

        for over in innings.get("overs", []):
            over_number = over.get("over")

            for delivery_number, delivery in enumerate(
                over.get("deliveries", []), start=1
            ):
                batter = delivery.get("batter")
                bowler = delivery.get("bowler")
                non_striker = delivery.get("non_striker")
                runs = delivery.get("runs", {}) or {}
                extras = delivery.get("extras", {}) or {}

                batter_runs = int(runs.get("batter", 0) or 0)
                extra_runs = int(runs.get("extras", 0) or 0)
                total_runs = int(runs.get("total", batter_runs + extra_runs) or 0)

                wides = int(extras.get("wides", 0) or 0)
                noballs = int(extras.get("noballs", 0) or 0)
                byes = int(extras.get("byes", 0) or 0)
                legbyes = int(extras.get("legbyes", 0) or 0)

                # Batting: create batter record even when the batter scores zero.
                if batter is not None:
                    if batter not in batting_stats:
                        batting_stats[batter] = {
                            "runs": 0, "balls_faced": 0,
                            "fours": 0, "sixes": 0,
                        }
                        first_seen_batter[batter] = len(first_seen_batter) + 1

                    batting_stats[batter]["runs"] += batter_runs

                    # Widest and no-balls are excluded from balls faced here.
                    if wides == 0 and noballs == 0:
                        batting_stats[batter]["balls_faced"] += 1

                    if batter_runs == 4 and not is_non_boundary(delivery):
                        batting_stats[batter]["fours"] += 1
                    if batter_runs == 6 and not is_non_boundary(delivery):
                        batting_stats[batter]["sixes"] += 1

                # Bowling: count every delivery bowled by the named bowler.
                if bowler is not None:
                    if bowler not in bowling_stats:
                        bowling_stats[bowler] = {
                            "legal_balls": 0, "runs_conceded": 0,
                            "wickets": 0, "wides": 0, "noballs": 0,
                            "dots": 0, "maidens": 0,
                        }

                    bowling_stats[bowler]["wides"] += wides
                    bowling_stats[bowler]["noballs"] += noballs

                    # Bowler is charged batter runs plus wides and no-balls.
                    bowling_stats[bowler]["runs_conceded"] += (
                        batter_runs + wides + noballs
                    )

                    is_legal = (wides == 0 and noballs == 0)
                    if is_legal:
                        bowling_stats[bowler]["legal_balls"] += 1
                        if total_runs == 0:
                            bowling_stats[bowler]["dots"] += 1

                # Process wicket events separately from the delivery batter.
                for wicket in delivery.get("wickets", []) or []:
                    player_out = wicket.get("player_out")
                    kind = str(wicket.get("kind", "")).strip().lower()

                    if player_out:
                        previous = wicket_details.get(
                            player_out,
                            {"dismissal_kinds": [], "was_dismissed": False}
                        )
                        previous["dismissal_kinds"].append(kind)
                        previous["was_dismissed"] = True
                        wicket_details[player_out] = previous

                    if (
                        bowler is not None
                        and kind not in NON_BOWLER_WICKET_KINDS
                    ):
                        bowling_stats[bowler]["wickets"] += 1

        # Create batting innings-level rows.
        for player_name, stats in batting_stats.items():
            dismissal = wicket_details.get(
                player_name,
                {"dismissal_kinds": [], "was_dismissed": False}
            )
            batting_innings_rows.append({
                **{k: v for k, v in context.items() if k != "registry_people"},
                "innings": innings_number,
                "team": batting_team,
                "opposition": opposition,
                "player_name": player_name,
                "player_id": player_id_for(player_name, registry_people),
                "player_name_norm": normalize_label(player_name),
                "team_norm": normalize_label(batting_team),
                "opposition_norm": normalize_label(opposition),
                **stats,
                "batting_position": first_seen_batter[player_name],
                "dismissal_kind": (
                    "|".join(dismissal["dismissal_kinds"])
                    if dismissal["dismissal_kinds"] else None
                ),
                "was_dismissed": dismissal["was_dismissed"],
            })

        # Create bowling innings-level rows.
        for player_name, stats in bowling_stats.items():
            legal_balls = stats["legal_balls"]
            bowling_innings_rows.append({
                **{k: v for k, v in context.items() if k != "registry_people"},
                "innings": innings_number,
                "team": other_team(
                    batting_team,
                    context["team_1"],
                    context["team_2"]
                ),
                "opposition": batting_team,
                "player_name": player_name,
                "player_id": player_id_for(player_name, registry_people),
                "player_name_norm": normalize_label(player_name),
                "team_norm": normalize_label(
                    other_team(batting_team, context["team_1"], context["team_2"])
                ),
                "opposition_norm": normalize_label(batting_team),
                **stats,
                "overs": f"{legal_balls // 6}.{legal_balls % 6}",
                "economy": (
                    stats["runs_conceded"] / (legal_balls / 6)
                    if legal_balls > 0 else np.nan
                ),
            })

    if file_number % 200 == 0:
        print(f"Processed {file_number}/{len(json_files)} matches")

print("Finished source extraction.")
print("Match rows:", len(match_rows))
print("Batting innings rows:", len(batting_innings_rows))
print("Bowling innings rows:", len(bowling_innings_rows))


Processed 200/1243 matches
Processed 400/1243 matches
Processed 600/1243 matches
Processed 800/1243 matches
Processed 1000/1243 matches
Processed 1200/1243 matches
Finished source extraction.
Match rows: 1243
Batting innings rows: 18842
Bowling innings rows: 14734


## 5. Create player-match datasets

The innings-level records are aggregated into one row per player per match.

- Runs, balls, boundaries, wickets, and runs conceded are summed.
- Match metadata is retained.
- Batting position is the minimum position across innings.
- Dismissal kinds are combined as a pipe-separated string when more than one dismissal event is recorded for the same player in a match.
- Bowling overs are recalculated from total legal balls, not summed as decimal values.


In [4]:
batting_innings = pd.DataFrame(batting_innings_rows)
bowling_innings = pd.DataFrame(bowling_innings_rows)
matches = pd.DataFrame(match_rows)
player_registry_observations = pd.DataFrame(player_registry_rows)

print("Batting innings shape:", batting_innings.shape)
print("Bowling innings shape:", bowling_innings.shape)
print("Match metadata shape:", matches.shape)

display(batting_innings.head())
display(bowling_innings.head())


Batting innings shape: (18842, 24)
Bowling innings shape: (14734, 26)
Match metadata shape: (1243, 9)


,match_id,source_file,date,season,venue,city,team_1,team_2,match_type,innings,...,player_name_norm,team_norm,opposition_norm,runs,balls_faced,fours,sixes,batting_position,dismissal_kind,was_dismissed
0,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,dawarner,sunrisershyderabad,royalchallengersbangalore,14,7,2,1,1,caught,True
1,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sdhawan,sunrisershyderabad,royalchallengersbangalore,40,31,5,0,2,caught,True
2,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,mchenriques,sunrisershyderabad,royalchallengersbangalore,52,37,3,2,3,caught,True
3,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,yuvrajsingh,sunrisershyderabad,royalchallengersbangalore,62,27,7,3,4,bowled,True
4,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,djhooda,sunrisershyderabad,royalchallengersbangalore,16,12,0,1,5,NaN,False


,match_id,source_file,date,season,venue,city,team_1,team_2,match_type,innings,...,opposition_norm,legal_balls,runs_conceded,wickets,wides,noballs,dots,maidens,overs,economy
0,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sunrisershyderabad,24,31,1,2,0,10,0,4.0,7.750000
1,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sunrisershyderabad,24,55,1,3,1,5,0,4.0,13.750000
2,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sunrisershyderabad,24,22,1,0,0,5,0,4.0,5.500000
3,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sunrisershyderabad,18,36,0,0,0,2,0,3.0,12.000000
4,1082591,1082591.json,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,None,1,...,sunrisershyderabad,18,41,0,0,0,3,0,3.0,13.666667


In [5]:
# Aggregation helpers.
metadata_cols = [
    "match_id", "source_file", "date", "season", "venue", "city",
    "team_1", "team_2", "match_type"
]

batting_group_cols = [
    "match_id", "date", "season", "venue", "city", "team_1", "team_2",
    "match_type", "team", "opposition", "player_name", "player_id",
    "player_name_norm", "team_norm", "opposition_norm"
]

batting_player_match = (
    batting_innings
    .groupby(batting_group_cols, dropna=False, as_index=False)
    .agg(
        runs=("runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum"),
        batting_position=("batting_position", "min"),
        was_dismissed=("was_dismissed", "max"),
        dismissal_kind=("dismissal_kind", lambda s: "|".join(
            sorted({str(v) for v in s.dropna() if str(v).strip()})
        ) or None),
        innings_count=("innings", "nunique"),
    )
)

batting_player_match["strike_rate"] = np.where(
    batting_player_match["balls_faced"] > 0,
    batting_player_match["runs"] * 100 / batting_player_match["balls_faced"],
    np.nan
)

bowling_group_cols = [
    "match_id", "date", "season", "venue", "city", "team_1", "team_2",
    "match_type", "team", "opposition", "player_name", "player_id",
    "player_name_norm", "team_norm", "opposition_norm"
]

bowling_player_match = (
    bowling_innings
    .groupby(bowling_group_cols, dropna=False, as_index=False)
    .agg(
        legal_balls=("legal_balls", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("wickets", "sum"),
        wides=("wides", "sum"),
        noballs=("noballs", "sum"),
        dots=("dots", "sum"),
        innings_count=("innings", "nunique"),
    )
)

bowling_player_match["overs"] = (
    (bowling_player_match["legal_balls"] // 6).astype(int).astype(str)
    + "."
    + (bowling_player_match["legal_balls"] % 6).astype(int).astype(str)
)

bowling_player_match["economy"] = np.where(
    bowling_player_match["legal_balls"] > 0,
    bowling_player_match["runs_conceded"] * 6
    / bowling_player_match["legal_balls"],
    np.nan
)

print("Batting player-match shape:", batting_player_match.shape)
print("Bowling player-match shape:", bowling_player_match.shape)

display(batting_player_match.head())
display(bowling_player_match.head())


Batting player-match shape: (18768, 24)
Bowling player-match shape: (14700, 24)


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,runs,balls_faced,fours,sixes,batting_position,was_dismissed,dismissal_kind,innings_count,strike_rate
0,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,6,2,0,1,11,False,NaN,1,300.000000
1,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,32,21,2,3,1,True,caught,1,152.380952
2,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,31,15,4,1,4,True,run out,1,206.666667
3,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,24,16,5,0,2,True,bowled,1,150.000000
4,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,0,2,0,0,8,True,bowled,1,0.000000


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,legal_balls,runs_conceded,wickets,wides,noballs,dots,innings_count,overs,economy
0,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,24,55,1,3,1,5,1,4.0,13.750000
1,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,18,36,0,0,0,2,1,3.0,12.000000
2,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,18,41,0,0,0,3,1,3.0,13.666667
3,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,6,10,1,0,0,1,1,1.0,10.000000
4,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,NaN,Royal Challengers Bangalore,Sunrisers Hyderabad,...,sunrisershyderabad,6,11,0,0,0,0,1,1.0,11.000000


## 6. Validate player-match dataset keys

A player-match dataset should have no more than one row for a player in a match after innings aggregation.

The source key uses `match_id` plus the source player identity. When registry IDs are missing, the source player name remains available and must not be replaced with an invented ID.


In [6]:
def duplicate_key_report(df, name):
    key = ["match_id", "player_name"]
    duplicates = df.duplicated(key, keep=False)
    print(f"{name}: rows={len(df):,}; duplicate match/player keys={int(duplicates.sum()):,}")
    if duplicates.any():
        display(df.loc[duplicates, key].sort_values(key).head(20))
    return not duplicates.any()


batting_unique_key = duplicate_key_report(
    batting_player_match, "Batting player-match"
)
bowling_unique_key = duplicate_key_report(
    bowling_player_match, "Bowling player-match"
)

assert batting_unique_key, "Batting player-match key is not unique."
assert bowling_unique_key, "Bowling player-match key is not unique."

print("Player-match key validation: PASS")


Batting player-match: rows=18,768; duplicate match/player keys=0
Bowling player-match: rows=14,700; duplicate match/player keys=0
Player-match key validation: PASS


## 7. Numeric and logical validation

These checks are not intended to claim that every scorecard is perfect. They detect obvious aggregation or parsing failures.

Potential edge cases are retained for investigation rather than silently corrected.


In [7]:
validation_results = []

def add_check(name, passed, detail):
    validation_results.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail),
    })

add_check(
    "Batting runs are non-negative",
    (batting_player_match["runs"] >= 0).all(),
    f"minimum={batting_player_match['runs'].min()}"
)
add_check(
    "Batting balls faced are non-negative",
    (batting_player_match["balls_faced"] >= 0).all(),
    f"minimum={batting_player_match['balls_faced'].min()}"
)
add_check(
    "Batting boundaries are non-negative",
    (
        (batting_player_match["fours"] >= 0)
        & (batting_player_match["sixes"] >= 0)
    ).all(),
    "fours and sixes checked"
)
add_check(
    "Bowling legal balls are non-negative",
    (bowling_player_match["legal_balls"] >= 0).all(),
    f"minimum={bowling_player_match['legal_balls'].min()}"
)
add_check(
    "Bowling runs conceded are non-negative",
    (bowling_player_match["runs_conceded"] >= 0).all(),
    f"minimum={bowling_player_match['runs_conceded'].min()}"
)
add_check(
    "Bowling wickets are non-negative",
    (bowling_player_match["wickets"] >= 0).all(),
    f"maximum={bowling_player_match['wickets'].max()}"
)
add_check(
    "Match IDs are populated in batting dataset",
    batting_player_match["match_id"].notna().all(),
    f"missing={batting_player_match['match_id'].isna().sum()}"
)
add_check(
    "Match IDs are populated in bowling dataset",
    bowling_player_match["match_id"].notna().all(),
    f"missing={bowling_player_match['match_id'].isna().sum()}"
)
add_check(
    "Dates parse in batting dataset",
    batting_player_match["date"].notna().all(),
    f"missing={batting_player_match['date'].isna().sum()}"
)
add_check(
    "Dates parse in bowling dataset",
    bowling_player_match["date"].notna().all(),
    f"missing={bowling_player_match['date'].isna().sum()}"
)

validation_df = pd.DataFrame(validation_results)
display(validation_df)

if not validation_df["passed"].all():
    print("WARNING: At least one validation failed. Investigate before continuing.")
else:
    print("Numeric/logical validation checks: PASS")


,check,passed,detail
0,Batting runs are non-negative,True,minimum=0
1,Batting balls faced are non-negative,True,minimum=0
2,Batting boundaries are non-negative,True,fours and sixes checked
3,Bowling legal balls are non-negative,True,minimum=0
4,Bowling runs conceded are non-negative,True,minimum=0
5,Bowling wickets are non-negative,True,maximum=6
6,Match IDs are populated in batting dataset,True,missing=0
7,Match IDs are populated in bowling dataset,True,missing=0
8,Dates parse in batting dataset,True,missing=0
9,Dates parse in bowling dataset,True,missing=0


Numeric/logical validation checks: PASS


## 8. Compare derived 2026 data with supplied scorecards

The 2026 scorecards are used as a validation/reference source. They are **not appended** to the Cricsheet historical datasets.

Comparison needs a reliable match-directory-to-Cricsheet-match mapping. Since source names and match identifiers can differ, the first step is to inspect actual scorecard schemas and sample rows. Any comparison must be keyed by date, teams, and a verified match mapping, not by row order.


In [8]:
batting_scorecard_paths = sorted(DATA_2026_DIR.rglob("batting_scorecard.csv"))
bowling_scorecard_paths = sorted(DATA_2026_DIR.rglob("bowling_scorecard.csv"))

print("2026 batting scorecard files:", len(batting_scorecard_paths))
print("2026 bowling scorecard files:", len(bowling_scorecard_paths))

if batting_scorecard_paths:
    print("\nBatting scorecard columns:")
    print(pd.read_csv(batting_scorecard_paths[0]).columns.tolist())
    display(pd.read_csv(batting_scorecard_paths[0]).head())

if bowling_scorecard_paths:
    print("\nBowling scorecard columns:")
    print(pd.read_csv(bowling_scorecard_paths[0]).columns.tolist())
    display(pd.read_csv(bowling_scorecard_paths[0]).head())


2026 batting scorecard files: 74
2026 bowling scorecard files: 74

Batting scorecard columns:
['innings', 'team', 'batter', 'runs', 'balls', 'fours', 'sixes', 'strike_rate', 'dismissal', 'batting_position']


,innings,team,batter,runs,balls,fours,sixes,strike_rate,dismissal,batting_position
0,1,Sunrisers Hyderabad,TM Head,11,9,2,0,122.22,c PD Salt b JA Duffy,1
1,1,Sunrisers Hyderabad,Abhishek Sharma,7,8,0,1,87.50,c JM Sharma b JA Duffy,2
2,1,Sunrisers Hyderabad,Ishan Kishan,80,38,8,5,210.53,c PD Salt b Abhinandan Singh,3
3,1,Sunrisers Hyderabad,Nithish Kumar Reddy,1,6,0,0,16.67,c Abhinandan Singh b JA Duffy,4
4,1,Sunrisers Hyderabad,H Klaasen,31,22,2,1,140.91,c PD Salt b R Shepherd,5



Bowling scorecard columns:
['innings', 'team', 'bowler', 'overs', 'maidens', 'runs', 'wickets', 'economy', 'dots', 'wides', 'noballs']


,innings,team,bowler,overs,maidens,runs,wickets,economy,dots,wides,noballs
0,1,Royal Challengers Bengaluru,Abhinandan Singh,3.0,0,38,1,12.67,7,0,0
1,1,Royal Challengers Bengaluru,B Kumar,4.0,0,31,1,7.75,7,2,0
2,1,Royal Challengers Bengaluru,JA Duffy,4.0,0,22,3,5.50,13,1,0
3,1,Royal Challengers Bengaluru,KH Pandya,2.0,0,26,0,13.00,1,0,0
4,1,Royal Challengers Bengaluru,R Shepherd,4.0,0,54,3,13.50,5,1,0


## 9. Save Day 3 processed datasets

The notebook writes four files:

- `batting_player_match.csv`
- `bowling_player_match.csv`
- `match_metadata.csv`
- `player_registry_observations.csv`

The registry observations file is a source audit trail, not yet a canonical player dimension. Multiple rows per player are expected because a player can appear in multiple matches and teams.


In [9]:
# Export normalized processed artifacts.
# Do not write to data/raw/.

batting_output = PROCESSED_DIR / "batting_player_match.csv"
bowling_output = PROCESSED_DIR / "bowling_player_match.csv"
matches_output = PROCESSED_DIR / "match_metadata.csv"
registry_output = PROCESSED_DIR / "player_registry_observations.csv"

batting_player_match.sort_values(
    ["date", "match_id", "player_name"]
).to_csv(batting_output, index=False)

bowling_player_match.sort_values(
    ["date", "match_id", "player_name"]
).to_csv(bowling_output, index=False)

matches.sort_values(
    ["date", "match_id"]
).to_csv(matches_output, index=False)

player_registry_observations.to_csv(
    registry_output,
    index=False
)

for path in [
    batting_output,
    bowling_output,
    matches_output,
    registry_output,
]:
    print(f"Saved {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")


Saved data\processed\batting_player_match.csv (4,529,086 bytes)
Saved data\processed\bowling_player_match.csv (3,410,075 bytes)
Saved data\processed\match_metadata.csv (143,556 bytes)
Saved data\processed\player_registry_observations.csv (1,583,770 bytes)


## 10. Write the data dictionary

The data dictionary documents the outputs for future notebooks, SQL tables, APIs, and Power BI.


In [10]:
data_dictionary = pd.DataFrame([
    ["batting_player_match.csv", "match_id", "Source match ID from Cricsheet JSON"],
    ["batting_player_match.csv", "date", "Match date parsed from Cricsheet metadata"],
    ["batting_player_match.csv", "season", "Season value from Cricsheet metadata, stored as text"],
    ["batting_player_match.csv", "team", "Team batting in the match"],
    ["batting_player_match.csv", "opposition", "Other team in the match"],
    ["batting_player_match.csv", "player_name", "Original batter name from source"],
    ["batting_player_match.csv", "player_id", "Cricsheet registry ID when available; otherwise null"],
    ["batting_player_match.csv", "runs", "Sum of runs credited to batter"],
    ["batting_player_match.csv", "balls_faced", "Deliveries faced excluding wides and no-balls under this implementation"],
    ["batting_player_match.csv", "fours/sixes", "Batter-scored boundaries, excluding marked non-boundary runs"],
    ["batting_player_match.csv", "strike_rate", "Runs / balls_faced * 100; null if balls faced is zero"],
    ["bowling_player_match.csv", "legal_balls", "Deliveries counted as legal; excludes wides and no-balls"],
    ["bowling_player_match.csv", "overs", "Cricket overs display derived from legal balls, e.g. 3.2 means 3 overs and 2 balls"],
    ["bowling_player_match.csv", "runs_conceded", "Batter runs plus wides and no-balls"],
    ["bowling_player_match.csv", "wickets", "Wickets credited to bowler under the explicit dismissal-kind rule"],
    ["bowling_player_match.csv", "economy", "Runs conceded * 6 / legal balls; null if no legal balls"],
    ["match_metadata.csv", "match_id", "Cricsheet JSON filename stem"],
    ["match_metadata.csv", "team_1/team_2", "Teams as listed in Cricsheet match metadata"],
    ["player_registry_observations.csv", "registry_id", "Cricsheet registry person ID when available"],
], columns=["file", "field", "definition"])

display(data_dictionary)
data_dictionary_path = DOCS_DIR / "day3_data_dictionary.csv"
data_dictionary.to_csv(data_dictionary_path, index=False)
print("Saved:", data_dictionary_path)


,file,field,definition
0,batting_player_match.csv,match_id,Source match ID from Cricsheet JSON
1,batting_player_match.csv,date,Match date parsed from Cricsheet metadata
2,batting_player_match.csv,season,"Season value from Cricsheet metadata, stored a..."
3,batting_player_match.csv,team,Team batting in the match
4,batting_player_match.csv,opposition,Other team in the match
5,batting_player_match.csv,player_name,Original batter name from source
6,batting_player_match.csv,player_id,Cricsheet registry ID when available; otherwis...
7,batting_player_match.csv,runs,Sum of runs credited to batter
8,batting_player_match.csv,balls_faced,Deliveries faced excluding wides and no-balls ...
9,batting_player_match.csv,fours/sixes,"Batter-scored boundaries, excluding marked non..."


Saved: d:\Git\Personal\ipl-player-performance\docs\day3_data_dictionary.csv


## 11. Document Day 3 decisions

The decisions below are written to `docs/day3_dataset_construction.md` for project traceability.


In [11]:
decision_text = f"""# Day 3: Player-Match Dataset Construction

## Outputs

- `data/processed/batting_player_match.csv`
- `data/processed/bowling_player_match.csv`
- `data/processed/match_metadata.csv`
- `data/processed/player_registry_observations.csv`
- `docs/day3_data_dictionary.csv`

## Source policy

Cricsheet JSON is the primary historical source. The supplied 2026 scorecards are used for comparison and validation, not automatically appended to the historical datasets.

## Aggregation keys

- Batting: one row per `match_id` + `player_name`.
- Bowling: one row per `match_id` + `player_name`.

Innings-level statistics are aggregated into player-match records.

## Identity policy

Original player names are preserved. Normalized names are comparison keys only. Player IDs come from the Cricsheet registry when available. Missing IDs are not fabricated.

## Metric definitions

- Batting runs: sum of `runs.batter`.
- Balls faced: excludes wides and no-balls under the current implementation.
- Fours and sixes: batter-scored boundaries, excluding marked non-boundary runs.
- Bowling runs conceded: batter runs + wides + no-balls.
- Legal balls: deliveries excluding wides and no-balls.
- Bowler wickets: excludes configured non-bowler-credit dismissal types.
- Overs: derived from legal balls. Use `legal_balls` for numeric calculations.

## Validation

- Match/player key uniqueness.
- Non-negative performance statistics.
- Populated match IDs and valid dates.
- Inspection of 2026 scorecard schemas.

## Limitations and next checks

1. Compare derived 2026 statistics with scorecards using a verified match mapping.
2. Investigate failed validations before proceeding.
3. Review dismissal-kind and non-boundary edge cases.
4. Confirm balls-faced and bowler-credit rules.
5. Next-match targets belong to Day 4.
6. No ML/DL model is trained on Day 3.

## Row counts

- Cricsheet JSON matches processed: {len(json_files):,}
- Batting player-match rows: {len(batting_player_match):,}
- Bowling player-match rows: {len(bowling_player_match):,}
- Match metadata rows: {len(matches):,}
"""

decision_path = DOCS_DIR / "day3_dataset_construction.md"
decision_path.write_text(decision_text, encoding="utf-8")

print("Saved:", decision_path)

Saved: d:\Git\Personal\ipl-player-performance\docs\day3_dataset_construction.md


# Day 3 completion checklist

Day 3 is complete only after all cells run and these outputs have been reviewed:

- [ ] `batting_player_match.csv` created
- [ ] `bowling_player_match.csv` created
- [ ] `match_metadata.csv` created
- [ ] `player_registry_observations.csv` created
- [ ] Player-match keys are unique
- [ ] Numeric/logical validations pass or exceptions are documented
- [ ] Scorecard schemas inspected
- [ ] `docs/day3_data_dictionary.csv` created
- [ ] `docs/day3_dataset_construction.md` created
- [ ] No raw files changed
- [ ] No ML/DL models trained

## Day 4 preview

Day 4 will create the targets:

- `next_match_runs`
- `next_match_wickets`
- `next_match_runs_conceded`

The targets will be created by sorting each player's match records chronologically and shifting the outcome from the next eligible appearance backward by one row. The target match's statistics will never be used as input features for that same match.
